# Pipeline-Based Baseline — Unified Dataset (Google Colab)

Implements a **true sequential pipeline** for sarcasm-aware cyberbullying detection:
- **Stage 1**: Train a BERT encoder for sarcasm detection
- **Stage 2**: **Freeze** the sarcasm encoder. Use its [CLS] representation + sarcasm probability as input to a harm classification head (no separate BERT for harm)

The harm classifier **fully depends** on the sarcasm encoder's representations, ensuring upstream sarcasm errors propagate into harm predictions.

Computes **Cascade Error Rate (CER)** to measure error propagation.

Tasks:
- **sarc**: binary (0/1) — sarcasm detection
- **intent**: binary (0/1) — harmful intent detection (uses `harm` column)

**Before running:**
1. Set runtime to GPU: `Runtime > Change runtime type > T4 GPU`
2. Upload `cyberbully_train_ready.csv` to Google Drive at: `MyDrive/mtl-bert/data/`
3. Run all cells in order.

Results are saved to Drive so they survive session disconnects.

In [ ]:
# Install dependencies
!pip install -q transformers scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR    = "/content/drive/MyDrive/mtl-bert"
DATA_DIR    = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "results", "unified-dataset", "pipeline-baseline")

DATA_PATH = os.path.join(DATA_DIR, "cyberbully_train_ready.csv")

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Base dir : {BASE_DIR}")
print(f"Data file: {DATA_PATH}")
print(f"Results  : {RESULTS_DIR}")
print(f"File exists: {os.path.exists(DATA_PATH)}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import random
import json
import csv
from collections import Counter
from typing import Dict, List

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Config ──

MODEL_NAME = "bert-base-uncased"
SEEDS      = [42, 123, 456]
BATCH_SIZE = 16
NUM_EPOCHS = 5
MAX_LENGTH = 128

EMOTION_CLASSES = ["sadness", "joy", "love", "anger", "fear", "surprise"]
EMOTION_TO_IDX  = {name: i for i, name in enumerate(EMOTION_CLASSES)}

config = {
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "max_length": MAX_LENGTH,
}

In [ ]:
# ── Helpers ──

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def compute_metrics(predictions, labels):
    return {
        "accuracy" : accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, average="weighted", zero_division=0),
        "recall"   : recall_score(labels, predictions, average="weighted", zero_division=0),
        "f1"       : f1_score(labels, predictions, average="weighted", zero_division=0),
    }

In [ ]:
# ── Data Loading ──

def load_unified_dataset(data_path):
    samples = []
    skipped = 0
    with open(data_path, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row in reader:
            text = (row.get("text", "") or "").strip()
            if not text:
                skipped += 1
                continue
            try:
                sarc = int(row["sarcasm"])
                intent = int(row["harm"])
            except (ValueError, KeyError):
                skipped += 1
                continue
            emotion_label = (row.get("emotion", "") or "").strip()
            if emotion_label not in EMOTION_TO_IDX:
                skipped += 1
                continue
            emotion = EMOTION_TO_IDX[emotion_label]
            samples.append({"text": text, "sarc": sarc, "intent": intent, "emotion": emotion})
    if skipped > 0:
        print(f"  Skipped {skipped} rows")
    return samples

print(f"Loading dataset: {DATA_PATH}")
all_samples = load_unified_dataset(DATA_PATH)
print(f"  Total valid samples: {len(all_samples)}")

sarc_dist   = Counter(s["sarc"] for s in all_samples)
intent_dist = Counter(s["intent"] for s in all_samples)
print(f"  Sarcasm:     No={sarc_dist[0]}, Yes={sarc_dist[1]}")
print(f"  Harm Intent: Not harmful={intent_dist[0]}, Harmful={intent_dist[1]}")

In [ ]:
# ── Datasets ──

class SingleTaskDataset(Dataset):
    def __init__(self, samples: List[Dict], task_name: str, tokenizer, max_length=128):
        self.tokenizer, self.max_length = tokenizer, max_length
        self.task_name, self.samples = task_name, samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        enc = self.tokenizer(s["text"], truncation=True, padding="max_length",
                             max_length=self.max_length, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "label": torch.tensor(s[self.task_name], dtype=torch.long)}


class PipelineFeatureDataset(Dataset):
    """Dataset of pre-extracted [CLS] embeddings + sarcasm probs for harm training."""
    def __init__(self, embeddings, sarc_probs, samples):
        self.embeddings = embeddings
        self.sarc_probs = sarc_probs
        self.labels = [s["intent"] for s in samples]
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            "cls_emb": self.embeddings[idx],
            "sarc_prob": torch.tensor(self.sarc_probs[idx], dtype=torch.float),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [ ]:
# ── Models ──

class SarcasmBERT(nn.Module):
    """Stage 1: BERT encoder fine-tuned for sarcasm detection."""
    def __init__(self, model_name: str):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(out.last_hidden_state[:, 0])

    def get_embedding_and_prediction(self, input_ids, attention_mask):
        """Return both [CLS] embedding and sarcasm probability."""
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = out.last_hidden_state[:, 0]
        logits = self.classifier(cls_emb)
        sarc_prob = torch.sigmoid(logits.squeeze(-1))
        return cls_emb, sarc_prob


class PipelineHarmHead(nn.Module):
    """
    Stage 2: Harm classification head on FROZEN sarcasm encoder output.
    Input: [CLS] embedding (768) + sarcasm probability (1) = 769
    No separate BERT encoder.
    """
    def __init__(self, input_size: int = 769, dropout: float = 0.3):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(input_size, input_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(input_size // 2, 1),
        )

    def forward(self, cls_emb, sarc_prob):
        combined = torch.cat([cls_emb, sarc_prob.unsqueeze(-1)], dim=-1)
        return self.head(combined)

In [ ]:
# ── Training Helpers ──

def train_sarcasm_model(model, train_loader, val_loader, device, num_epochs=5):
    """Train the sarcasm BERT encoder (Stage 1)."""
    optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    loss_fn = nn.BCEWithLogitsLoss()
    best_val_f1, best_state = 0.0, None

    for epoch in range(num_epochs):
        model.train()
        total_loss, n_batches = 0.0, 0
        for batch in train_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device).float()
            logits = model(ids, mask).squeeze(-1)
            loss = loss_fn(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                ids = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                logits = model(ids, mask).squeeze(-1)
                preds = (torch.sigmoid(logits) > 0.5).long()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(batch["label"].numpy())

        val_m = compute_metrics(all_preds, all_labels)
        print(f"    Epoch {epoch+1}/{num_epochs} - Loss: {total_loss/max(n_batches,1):.4f} - "
              f"Val Acc: {val_m['accuracy']:.4f}, Val F1: {val_m['f1']:.4f}")
        if val_m["f1"] > best_val_f1:
            best_val_f1 = val_m["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    return best_state, best_val_f1


def extract_pipeline_features(sarc_model, samples, tokenizer, device,
                              max_length=128, batch_size=32):
    """Extract [CLS] embeddings and sarcasm probs from frozen sarcasm encoder."""
    sarc_model.eval()
    all_embeddings, all_sarc_probs = [], []
    texts = [s["text"] for s in samples]
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i+batch_size], truncation=True,
                        padding="max_length", max_length=max_length,
                        return_tensors="pt")
        with torch.no_grad():
            cls_emb, sarc_prob = sarc_model.get_embedding_and_prediction(
                enc["input_ids"].to(device), enc["attention_mask"].to(device))
        all_embeddings.append(cls_emb.cpu())
        all_sarc_probs.extend(sarc_prob.cpu().numpy().tolist())
    return torch.cat(all_embeddings, dim=0), all_sarc_probs


def train_harm_head(harm_head, train_loader, val_loader, device, num_epochs=5):
    """Train the harm head on frozen sarcasm encoder features (Stage 2)."""
    optimizer = optim.AdamW(harm_head.parameters(), lr=2e-5, weight_decay=0.01)
    loss_fn = nn.BCEWithLogitsLoss()
    best_val_f1, best_state = 0.0, None

    for epoch in range(num_epochs):
        harm_head.train()
        total_loss, n_batches = 0.0, 0
        for batch in train_loader:
            cls_emb = batch["cls_emb"].to(device)
            sarc_prob = batch["sarc_prob"].to(device)
            labels = batch["label"].to(device).float()
            logits = harm_head(cls_emb, sarc_prob).squeeze(-1)
            loss = loss_fn(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(harm_head.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

        harm_head.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                cls_emb = batch["cls_emb"].to(device)
                sarc_prob = batch["sarc_prob"].to(device)
                logits = harm_head(cls_emb, sarc_prob).squeeze(-1)
                preds = (torch.sigmoid(logits) > 0.5).long()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(batch["label"].numpy())

        val_m = compute_metrics(all_preds, all_labels)
        print(f"    Epoch {epoch+1}/{num_epochs} - Loss: {total_loss/max(n_batches,1):.4f} - "
              f"Val Acc: {val_m['accuracy']:.4f}, Val F1: {val_m['f1']:.4f}")
        if val_m["f1"] > best_val_f1:
            best_val_f1 = val_m["f1"]
            best_state = {k: v.cpu().clone() for k, v in harm_head.state_dict().items()}

    return best_state, best_val_f1


def evaluate_harm_head(harm_head, dataloader, device):
    """Evaluate the harm classification head."""
    harm_head.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            cls_emb = batch["cls_emb"].to(device)
            sarc_prob = batch["sarc_prob"].to(device)
            logits = harm_head(cls_emb, sarc_prob).squeeze(-1)
            preds = (torch.sigmoid(logits) > 0.5).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch["label"].numpy())
    return compute_metrics(all_preds, all_labels), all_preds, all_labels


def compute_cer(preds_with, preds_without, labels):
    total = len(labels)
    caused = sum(1 for i in range(total)
                 if preds_with[i] != labels[i] and preds_without[i] == labels[i])
    return caused / total if total > 0 else 0.0

In [ ]:
# ── Main Training ──

print("=" * 60)
print("  Pipeline-Based Baseline (Unified Dataset)")
print("=" * 60)

print(f"\nLoading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Resume support
progress_path = os.path.join(RESULTS_DIR, "training_progress.json")
all_results = {"sarc": [], "intent": [], "cer": []}
completed = set()

if os.path.exists(progress_path):
    with open(progress_path, "r") as f:
        progress = json.load(f)
    completed = set(progress.get("completed", []))
    all_results = progress.get("results", all_results)
    if completed:
        print(f"\nResuming: {len(completed)} seed(s) already done.")

for seed_idx, seed in enumerate(SEEDS):
    seed_key = f"seed{seed}"
    if seed_key in completed:
        print(f"\n--- Skipping seed {seed} (already done) ---")
        continue

    print(f"\n{'='*60}")
    print(f"  Seed {seed_idx+1}/{len(SEEDS)} (seed={seed})")
    print(f"{'='*60}")
    set_seed(seed)

    # Shared 80/10/10 split
    shuffled = all_samples.copy()
    random.shuffle(shuffled)
    n = len(shuffled)
    s1, s2 = int(0.8 * n), int(0.9 * n)
    train_samples = shuffled[:s1]
    val_samples   = shuffled[s1:s2]
    test_samples  = shuffled[s2:]
    print(f"  Split: Train={len(train_samples)}, Val={len(val_samples)}, Test={len(test_samples)}")

    # ── Stage 1: Train sarcasm encoder ──
    print(f"\n  [Stage 1] Training sarcasm encoder...")
    sarc_model = SarcasmBERT(MODEL_NAME).to(device)
    sarc_train_dl = DataLoader(SingleTaskDataset(train_samples, "sarc", tokenizer, MAX_LENGTH),
                               batch_size=BATCH_SIZE, shuffle=True)
    sarc_val_dl   = DataLoader(SingleTaskDataset(val_samples, "sarc", tokenizer, MAX_LENGTH),
                               batch_size=BATCH_SIZE, shuffle=False)
    sarc_test_dl  = DataLoader(SingleTaskDataset(test_samples, "sarc", tokenizer, MAX_LENGTH),
                               batch_size=BATCH_SIZE, shuffle=False)

    best_sarc_state, _ = train_sarcasm_model(sarc_model, sarc_train_dl, sarc_val_dl,
                                              device, num_epochs=NUM_EPOCHS)
    sarc_model.load_state_dict(best_sarc_state)
    sarc_model.to(device)

    # Evaluate sarcasm on test
    sarc_model.eval()
    sarc_preds, sarc_labels = [], []
    with torch.no_grad():
        for batch in sarc_test_dl:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            logits = sarc_model(ids, mask).squeeze(-1)
            preds = (torch.sigmoid(logits) > 0.5).long()
            sarc_preds.extend(preds.cpu().numpy())
            sarc_labels.extend(batch["label"].numpy())

    sarc_test_m = compute_metrics(sarc_preds, sarc_labels)
    print(f"    Sarcasm test: Acc={sarc_test_m['accuracy']:.4f}, F1={sarc_test_m['f1']:.4f}")

    # ── Stage 2: Extract features from FROZEN sarcasm encoder, train harm head ──
    print(f"\n  [Stage 2] Extracting features from frozen sarcasm encoder...")

    sarc_model.eval()
    for param in sarc_model.parameters():
        param.requires_grad = False

    train_embs, train_sarc_probs = extract_pipeline_features(
        sarc_model, train_samples, tokenizer, device, MAX_LENGTH)
    val_embs, val_sarc_probs = extract_pipeline_features(
        sarc_model, val_samples, tokenizer, device, MAX_LENGTH)
    test_embs, test_sarc_probs = extract_pipeline_features(
        sarc_model, test_samples, tokenizer, device, MAX_LENGTH)

    print(f"    Sarcasm positive rate on test: {np.mean([1 if p > 0.5 else 0 for p in test_sarc_probs]):.2%}")

    harm_train_dl = DataLoader(PipelineFeatureDataset(train_embs, train_sarc_probs, train_samples),
                               batch_size=BATCH_SIZE, shuffle=True)
    harm_val_dl   = DataLoader(PipelineFeatureDataset(val_embs, val_sarc_probs, val_samples),
                               batch_size=BATCH_SIZE, shuffle=False)
    harm_test_dl  = DataLoader(PipelineFeatureDataset(test_embs, test_sarc_probs, test_samples),
                               batch_size=BATCH_SIZE, shuffle=False)

    print(f"\n  [Stage 2] Training harm classification head...")
    harm_head = PipelineHarmHead(input_size=769).to(device)
    best_harm_state, _ = train_harm_head(harm_head, harm_train_dl, harm_val_dl,
                                          device, num_epochs=NUM_EPOCHS)
    harm_head.load_state_dict(best_harm_state)
    harm_head.to(device)

    # ── CER Analysis ──
    print(f"\n  [CER Analysis] Computing Cascade Error Rate...")

    harm_metrics, preds_with, harm_labels = evaluate_harm_head(harm_head, harm_test_dl, device)
    print(f"    Harm test (pipeline): Acc={harm_metrics['accuracy']:.4f}, F1={harm_metrics['f1']:.4f}")

    # With sarcasm zeroed out
    harm_test_dl_clean = DataLoader(
        PipelineFeatureDataset(test_embs, [0.0]*len(test_samples), test_samples),
        batch_size=BATCH_SIZE, shuffle=False)
    harm_metrics_clean, preds_without, _ = evaluate_harm_head(harm_head, harm_test_dl_clean, device)
    print(f"    Harm test (no sarc):  Acc={harm_metrics_clean['accuracy']:.4f}, F1={harm_metrics_clean['f1']:.4f}")

    cer = compute_cer(preds_with, preds_without, harm_labels)
    print(f"    CER = {cer:.4f} ({cer:.2%} of predictions corrupted by sarcasm)")

    total = len(harm_labels)
    print(f"    Errors with sarcasm: {sum(1 for i in range(total) if preds_with[i] != harm_labels[i])}/{total}")
    print(f"    Errors without sarcasm: {sum(1 for i in range(total) if preds_without[i] != harm_labels[i])}/{total}")

    # Save seed results
    all_results["sarc"].append(sarc_test_m)
    all_results["intent"].append(harm_metrics)
    all_results["cer"].append({"cer": cer,
                               "harm_f1_with_sarc": harm_metrics["f1"],
                               "harm_f1_without_sarc": harm_metrics_clean["f1"]})
    completed.add(seed_key)

    with open(progress_path, "w") as f:
        json.dump({"completed": list(completed), "results": all_results}, f, indent=2)
    print(f"  Progress saved ({len(completed)}/{len(SEEDS)} seeds done)")

In [ ]:
# ── Aggregated Results ──

print(f"\n{'='*60}")
print(f"  Pipeline Baseline: Aggregated Results ({len(SEEDS)} seeds)")
print(f"{'='*60}")

aggregated = {}
for task in ["sarc", "intent"]:
    agg = {}
    for metric in ["accuracy", "precision", "recall", "f1"]:
        values = [m[metric] for m in all_results[task]]
        agg[metric] = {"mean": float(np.mean(values)), "std": float(np.std(values))}
    aggregated[task] = agg
    print(f"\n  {task}:")
    for metric in ["accuracy", "precision", "recall", "f1"]:
        print(f"    {metric:>10s}: {agg[metric]['mean']:.4f} +/- {agg[metric]['std']:.4f}")

cer_values = [c["cer"] for c in all_results["cer"]]
aggregated["cer"] = {
    "mean": float(np.mean(cer_values)),
    "std": float(np.std(cer_values)),
    "per_seed": all_results["cer"],
}
print(f"\n  Cascade Error Rate (CER):")
print(f"    CER: {np.mean(cer_values):.4f} +/- {np.std(cer_values):.4f}")
print(f"    (MTL CER = 0 by design, no sequential dependency)")

aggregated["emotion_note"] = (
    "Emotion is trained independently in the pipeline (same as single-task "
    "baseline). Refer to STL baseline results for emotion metrics."
)

agg_path = os.path.join(RESULTS_DIR, "aggregated_results.json")
with open(agg_path, "w") as f:
    json.dump(aggregated, f, indent=2)
print(f"\nResults saved to {agg_path}")

# Summary
print(f"\n{'='*60}")
print(f"  Summary")
print(f"{'='*60}")
print(f"  Dataset: cyberbully_train_ready.csv (unified)")
print(f"  Total samples: {len(all_samples)}")
print(f"  Pipeline: sarcasm encoder (frozen) -> harm head")
print(f"  CER: {np.mean(cer_values):.4f} "
      f"(sarcasm errors corrupt {np.mean(cer_values):.2%} of harm predictions)")
print(f"  MTL CER: 0.0000 (no sequential dependency by architecture)")
print(f"\n  Done!")